[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_05_amplitude_equations_patterns.ipynb)


# 05. Amplitude equations and pattern selection

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** Why do very different systems share the same near-threshold equations?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- use symmetry to constrain an amplitude equation
- derive saturation amplitude and nonlinear frequency shift
- interpret Rayleigh–Bénard convection as an open dissipative structure and identify its control and order parameters
- simulate finite-wavenumber pattern selection and test numerical convergence


## Prerequisite bridge: Fourier modes and dispersion relations

A spatial field can be decomposed into Fourier modes $u(x,t)=\sum_k\hat u_k(t)e^{ikx}$. Because $\partial_x^2e^{ikx}=-k^2e^{ikx}$, linear constant-coefficient PDEs evolve each wavenumber independently. Substitution of $u\propto e^{\lambda(k)t+ikx}$ gives a **dispersion relation** $\lambda(k)$; modes with $\operatorname{Re}\lambda(k)>0$ grow.

The wavelength is $2\pi/k$. A finite periodic domain admits only discrete wavenumbers, and a numerical grid resolves only wavelengths sufficiently larger than its spacing. A complex amplitude $A=Re^{i\phi}$ conveniently stores both pattern strength $R$ and phase or position $\phi$.


## Universality near threshold

A complex amplitude $A$ for an oscillatory or traveling mode often obeys

$$\dot A=(\mu+i\omega_0)A-(g_r+i g_i)|A|^2A.$$

Phase symmetry $A\mapsto Ae^{i\phi}$ forbids constant and quadratic terms. For $\mu>0$ and $g_r>0$, the stable amplitude is $|A|=\sqrt{\mu/g_r}$; $g_i$ shifts the nonlinear frequency.

A minimal field equation for stationary finite-wavenumber patterns is the Swift–Hohenberg model,

$$\partial_tu=ru-(1+\partial_x^2)^2u-u^3,$$

whose linear growth rate is $\sigma(k)=r-(1-k^2)^2$. The band near $k=1$ becomes unstable first.


## Canonical dissipative structure: Rayleigh–Bénard convection

A horizontal fluid layer heated from below and cooled from above is continuously driven by a heat current and continuously dissipates energy through viscosity and thermal diffusion. Its reference state is motionless conduction, $\mathbf u=0$, with an approximately linear temperature profile. Buoyancy amplifies a vertical displacement because a warm parcel is lighter than its surroundings; viscosity and thermal diffusion damp it. Their competition is measured by

$$Ra=\frac{g\alpha\,\Delta T\,d^3}{\nu\kappa},\qquad Pr=\frac{\nu}{\kappa},$$

where $d$ is the layer depth, $\alpha$ the thermal-expansion coefficient, $\nu$ the kinematic viscosity, and $\kappa$ the thermal diffusivity. For an ideal laterally extended layer between rigid, isothermal plates, conduction loses stability at $Ra_c\simeq1708$ and a roll pair with horizontal period about $2.016d$ bifurcates supercritically. This is a particularly transparent **dissipative structure**: the ordered velocity and temperature fields exist only while the imposed temperature difference maintains the energy throughput.

Near onset, the signed amplitude $A$ of the critical roll mode obeys the same symmetry-constrained normal form encountered throughout synergetics,

$$\tau_A\dot A=\epsilon A-g_AA^3,\qquad \epsilon=\frac{Ra-Ra_c}{Ra_c},$$

so $A=0$ is stable below threshold and $|A|\propto\sqrt{\epsilon}$ above it. The sign distinguishes symmetry-related circulation orientations. Fast thermal and velocity modes are slaved to $A$, while the excess heat transport satisfies $Nu-1\propto A^2$ sufficiently close to onset. The visualization below is a weakly nonlinear critical-mode reconstruction, not a direct numerical solution of the full Oberbeck–Boussinesq equations. Module 09 later shows how Lorenz's three-mode truncation connects this same experiment to deterministic chaos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

# Near-onset critical-mode reconstruction for Rayleigh-Benard rolls.
Ra_c, g_landau = 1708.0, 1.0
epsilon = np.linspace(-0.25, 0.60, 400)
A_branch = np.sqrt(np.maximum(epsilon, 0.0)/g_landau)

x_rb = np.linspace(0, 2.016, 150)
z_rb = np.linspace(0, 1, 90)
X_rb, Z_rb = np.meshgrid(x_rb, z_rb)
k_rb = 2*np.pi/2.016

fig, ax = plt.subplots(1, 3, figsize=(13, 3.7))
for axis, eps in zip(ax[:2], [-0.10, 0.20]):
    A = np.sqrt(max(eps, 0.0)/g_landau)
    psi = A*np.sin(k_rb*X_rb)*np.sin(np.pi*Z_rb)
    u_horizontal = A*np.pi*np.sin(k_rb*X_rb)*np.cos(np.pi*Z_rb)
    w_vertical = -A*k_rb*np.cos(k_rb*X_rb)*np.sin(np.pi*Z_rb)
    theta = 0.14*A*np.cos(k_rb*X_rb)*np.sin(np.pi*Z_rb)
    temperature = 1-Z_rb+theta
    filled = axis.contourf(X_rb, Z_rb, temperature, levels=24, cmap="coolwarm")
    if A > 0:
        axis.streamplot(x_rb, z_rb, u_horizontal, w_vertical,
                        color="k", density=0.75, linewidth=0.6, arrowsize=0.7)
    state = "conduction" if eps < 0 else "convection rolls"
    axis.set(xlabel=r"$x/d$", ylabel=r"$z/d$",
             title=fr"{state}: $\epsilon={eps:+.2f}$")
    axis.set_aspect("equal")

ax[2].plot(epsilon, A_branch, label=r"$+A_*$")
ax[2].plot(epsilon, -A_branch, label=r"$-A_*$")
ax[2].axvline(0, color="k", lw=1)
ax[2].set(xlabel=r"$\epsilon=(Ra-Ra_c)/Ra_c$", ylabel="roll amplitude A",
          title="Supercritical onset")
ax[2].legend()
fig.colorbar(filled, ax=ax[:2], shrink=0.82, label="dimensionless temperature")
plt.show()

Nu_minus_1 = A_branch**2
print("Near onset: A scales as sqrt(epsilon), while Nu - 1 scales as epsilon.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

# Semi-implicit Fourier integration of the 1D Swift-Hohenberg equation.
N, L, dt, steps = 256, 80.0, 0.15, 1600
x = np.linspace(0, L, N, endpoint=False)
k = 2*np.pi*np.fft.fftfreq(N, d=L/N)
r = 0.35
linear = r - (1-k**2)**2
u = 0.05*rng.standard_normal(N)

snapshots = []
for n in range(steps):
    u_hat = np.fft.fft(u)
    nonlinear_hat = np.fft.fft(u**3)
    u_hat = (u_hat - dt*nonlinear_hat) / (1 - dt*linear)
    u = np.fft.ifft(u_hat).real
    if n in [0, 100, 400, steps-1]:
        snapshots.append((n*dt, u.copy()))

for t, field in snapshots:
    plt.plot(x, field, label=f"t={t:.1f}")
plt.xlabel("x")
plt.ylabel("u")
plt.title("Finite-wavenumber pattern selection")
plt.legend(ncol=2)
plt.show()

power = np.abs(np.fft.rfft(u))**2
k_pos = 2*np.pi*np.fft.rfftfreq(N, d=L/N)
k_peak = k_pos[1:][np.argmax(power[1:])]
print(f"dominant wavenumber = {k_peak:.3f}; linear prediction near 1")


## Numerical caution

Pattern-forming PDEs contain multiple scales and stiff linear operators. A convincing result reports domain size, grid spacing, time step, boundary conditions, transient length, and a convergence comparison. Near threshold, finite domains quantize allowed wavenumbers, so the observed $k$ need not equal the infinite-domain optimum.


## Assessed exercise

Repeat the Swift–Hohenberg simulation for three values of $r$ near threshold and two domain lengths. Measure saturated RMS amplitude and dominant wavenumber. Test the predicted amplitude exponent $1/2$ and explain finite-domain deviations.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** Haken, *Synergetics: Introduction and Advanced Topics*; Cross and Hohenberg, pattern selection and amplitude equations.

**Verified web reading:**
- [Rayleigh–Bénard convection overview](https://en.wikipedia.org/wiki/Rayleigh%E2%80%93B%C3%A9nard_convection)
- [Wen, Goluskin & Doering: steady convection rolls](https://doi.org/10.1017/jfm.2021.1042)
- [Cross & Hohenberg review](https://doi.org/10.1103/RevModPhys.65.851)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
